# Riemannian Embedded Transformer ✅

In [45]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np
import time
from nltk.corpus import wordnet as wn

class PoincareManifold:
    def __init__(self, c=1.0):
        self.c = c

    def mobius_add(self, x, y):
        x2, y2 = torch.sum(x*x, -1, True), torch.sum(y*y, -1, True)
        xy = torch.sum(x*y, -1, True)
        num = (1 + 2*self.c*xy + self.c*y2)*x + (1 - self.c*x2)*y
        denom = 1 + 2*self.c*xy + self.c**2 * x2*y2
        return num / torch.clamp(denom, min=1e-15)

    def exp_map(self, p, v):
        v_norm = torch.norm(v, 2, -1, True).clamp(min=1e-10)
        sqrt_c = np.sqrt(self.c)
        if isinstance(p, (int, float)) and p == 0:
            return torch.tanh(sqrt_c * v_norm) * v / (sqrt_c * v_norm)
        p_norm_sq = torch.sum(p*p, -1, True)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        v_at_origin = torch.tanh(sqrt_c * lambda_p * v_norm / 2) * v / (sqrt_c * v_norm)
        return self.mobius_add(p, v_at_origin)

    def log_map(self, p, x):
        p_tensor = torch.zeros_like(x) if (isinstance(p, (int, float)) and p == 0) else p
        x_trans = self.mobius_add(-p_tensor, x)
        x_norm = torch.norm(x_trans, 2, -1, True).clamp(min=1e-10)
        sqrt_c, lambda_p = np.sqrt(self.c), 2 / (1 - self.c * torch.sum(p_tensor*p_tensor, -1, True)).clamp(min=1e-15)
        scalar = (2 / (lambda_p * sqrt_c * x_norm)) * torch.atanh((sqrt_c * x_norm).clamp(max=1-1e-7))
        return scalar * x_trans

    def dist(self, x, y):
        sq_dist = torch.norm(x - y, 2, -1)**2
        nx_v, ny_v = 1 - self.c * torch.norm(x, 2, -1)**2, 1 - self.c * torch.norm(y, 2, -1)**2
        val = 1 + 2 * self.c * sq_dist / (nx_v.clamp(min=1e-15) * ny_v.clamp(min=1e-15))
        return (1 / np.sqrt(self.c)) * torch.acosh(val.clamp(min=1 + 1e-7))

    def project(self, x):
        norm = torch.norm(x, 2, -1, True)
        max_n = 0.99 / np.sqrt(self.c)
        return torch.where(norm >= max_n, x / norm * max_n, x)

class MultiHeadPoincareAttention(nn.Module):
    def __init__(self, d, h, manifold):
        super().__init__()
        self.h, self.d_h, self.manifold = h, d // h, manifold
        self.q_lin, self.k_lin, self.v_lin = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.out_proj = nn.Linear(d, d)

    def forward(self, x):
        b, s, d = x.shape
        q = self.manifold.exp_map(0, self.q_lin(x).view(b, s, self.h, self.d_h)).transpose(1, 2)
        k = self.manifold.exp_map(0, self.k_lin(x).view(b, s, self.h, self.d_h)).transpose(1, 2)
        v = self.v_lin(x).view(b, s, self.h, self.d_h).transpose(1, 2)
        attn = torch.softmax(-self.manifold.dist(q.unsqueeze(3), k.unsqueeze(2)) / 1.0, -1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(b, s, d)
        return self.out_proj(self.manifold.exp_map(0, out))

class PoincareTransformerBlock(nn.Module):
    def __init__(self, d, h, manifold):
        super().__init__()
        self.attn, self.ln1 = MultiHeadPoincareAttention(d, h, manifold), nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
        self.ln2, self.manifold = nn.LayerNorm(d), manifold

    def forward(self, x):
        res_attn = self.attn(self.ln1(x))
        x = self.manifold.exp_map(0, self.manifold.log_map(0, x) + res_attn)
        res_ff = self.ff(self.ln2(x))
        x = self.manifold.exp_map(0, self.manifold.log_map(0, x) + res_ff)
        return x

class RSGD(torch.optim.Optimizer):
    def __init__(self, params, lr, manifold):
        super().__init__(params, dict(lr=lr, manifold=manifold))
    def step(self):
        for group in self.param_groups:
            m, lr = group['manifold'], group['lr']
            for p in group['params']:
                if p.grad is None: continue
                p.data = m.project(m.exp_map(p.data, -lr * p.grad.data * ((1 - m.c * torch.sum(p.data**2, -1, True))**2 / 4)))

def evaluate_comprehensive(model, manifold, G, vocab_size, emb_layer):
    print("\nStarting Comprehensive Evaluation Suite...")
    start_time = time.time()
    
    with torch.no_grad():
        x_ball = manifold.exp_map(0, emb_layer(torch.arange(vocab_size)).unsqueeze(0))
        final = model(x_ball).squeeze(0)
        elapsed = time.time() - start_time
        throughput = vocab_size / (elapsed + 1e-9)

        p_dists, g_dists = [], []
        for _ in range(1000):
            u, v = np.random.choice(vocab_size, 2, replace=False)
            try:
                g_dists.append(nx.shortest_path_length(G, u, v))
                p_dists.append(manifold.dist(final[u], final[v]).item())
            except: continue
        correlation = np.corrcoef(g_dists, p_dists)[0, 1]
        max_radius = torch.norm(final, p=2, dim=-1).max().item()

        active_dims = (torch.var(final, dim=0) > 1e-4).sum().item()

        edges = list(G.edges())
        distortions = [abs(manifold.dist(final[u], final[v]).item() - 1.0) for u, v in edges]
        avg_distortion = np.mean(distortions)

        stability_scores = []
        for node in G.nodes():
            if G.degree(node) == 1 and node != 0:
                parent = list(G.neighbors(node))[0]
                siblings = [n for n in G.neighbors(parent) if n != node]
                if siblings:
                    d_p = manifold.dist(final[node], final[parent]).item()
                    d_s = np.mean([manifold.dist(final[node], final[s]).item() for s in siblings])
                    stability_scores.append(d_s / (d_p + 1e-6))
        avg_stability = np.mean(stability_scores) if stability_scores else 0.0

    return {
        "Correlation": round(correlation, 4),
        "Distortion": round(avg_distortion, 4),
        "Stability": round(avg_stability, 4),
        "Max_Radius": round(max_radius, 4),
        "Active_Dims": active_dims,
        "Throughput": int(throughput)
    }

G = nx.balanced_tree(r=3, h=4)
edge_index = torch.tensor(list(G.edges()), dtype=torch.long)
vocab_size, embed_dim, num_heads = len(G.nodes()), 16, 4
manifold = PoincareManifold()

emb_layer = nn.Embedding(vocab_size, embed_dim)
with torch.no_grad(): emb_layer.weight.data.uniform_(-0.02, 0.02)

model = PoincareTransformerBlock(embed_dim, num_heads, manifold)
optimizer = RSGD(list(emb_layer.parameters()) + list(model.parameters()), lr=0.5, manifold=manifold)

print(f"Training on Balanced Tree (N={vocab_size})...")
start_train_time = time.time()

for epoch in range(1201):
    epoch_start = time.time()
    
    optimizer.zero_grad()
    x_ball = manifold.exp_map(0, emb_layer(torch.arange(vocab_size)).unsqueeze(0))
    res = model(x_ball)
    
    u, v = edge_index[:, 0], edge_index[:, 1]
    pos_dist = manifold.dist(res[0, u], res[0, v])
    neg_dist = manifold.dist(res[0, u], res[0, torch.randint(0, vocab_size, (u.size(0),))])
    root_penalty = manifold.dist(res[0, 0], torch.zeros_like(res[0, 0])).mean()
    
    loss = pos_dist.mean() - 2.0 * torch.log(neg_dist.mean() + 1e-6) + 1.0 * root_penalty
    loss.backward()
    optimizer.step()
    
    if epoch == 800:
        for g in optimizer.param_groups: g['lr'] *= 0.1
        

    if epoch % 300 == 0:
        current_elapsed = time.time() - start_train_time
        avg_time_per_epoch = current_elapsed / (epoch + 1)
        print(f"Epoch {epoch:4d} | Loss: {loss.item():.4f} | Total Time: {current_elapsed:.2f}s | Avg: {avg_time_per_epoch:.4f}s/epoch")

total_train_time = time.time() - start_train_time
print(f"\nTraining Complete! Total duration: {total_train_time:.2f} seconds.")

results = evaluate_comprehensive(model, manifold, G, vocab_size, emb_layer)

print("\n")
print("FINAL RIEMANNIAN TRANSFORMER ANALYSIS")
print("~~"*20)
for k, v in results.items():
    print(f"{k:15}: {v}")
print("~~"*20)

Training on Balanced Tree (N=121)...
Epoch    0 | Loss: 2.8181 | Total Time: 0.02s | Avg: 0.0166s/epoch
Epoch  300 | Loss: -2.0679 | Total Time: 4.59s | Avg: 0.0153s/epoch
Epoch  600 | Loss: -2.5825 | Total Time: 9.29s | Avg: 0.0155s/epoch
Epoch  900 | Loss: -3.2465 | Total Time: 13.83s | Avg: 0.0153s/epoch
Epoch 1200 | Loss: -3.3767 | Total Time: 18.30s | Avg: 0.0152s/epoch

Training Complete! Total duration: 18.30 seconds.

Starting Comprehensive Evaluation Suite...


FINAL RIEMANNIAN TRANSFORMER ANALYSIS
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Correlation    : 0.5972
Distortion     : 1.1241
Stability      : 3.3926
Max_Radius     : 0.9977
Active_Dims    : 16
Throughput     : 121000000000
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


# Euclidian Embedded Transformer ✅

In [47]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np
import time

G = nx.balanced_tree(r=3, h=4)
edge_index = torch.tensor(list(G.edges()), dtype=torch.long)
vocab_size = len(G.nodes())
embed_dim, num_heads = 16, 4

class EuclideanTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=num_heads, 
            dim_feedforward=4*embed_dim,
            batch_first=True,
            norm_first=True
        )

    def forward(self, x_idx):
        x = self.embedding(x_idx)
        return self.encoder_layer(x)

def evaluate_comprehensive_euclidean(model, G, vocab_size):
    print("\nStarting Comprehensive Evaluation (Euclidean Baseline)...")
    start_time = time.time()
    model.eval()
    
    with torch.no_grad():
        all_indices = torch.arange(vocab_size).unsqueeze(0)
        final = model(all_indices).squeeze(0)
        elapsed = time.time() - start_time
        throughput = vocab_size / (elapsed + 1e-9)

        p_dists, g_dists = [], []
        for _ in range(1000):
            u, v = np.random.choice(vocab_size, 2, replace=False)
            try:
                g_dists.append(nx.shortest_path_length(G, int(u), int(v)))
                p_dists.append(torch.norm(final[u] - final[v], p=2).item())
            except: continue
        correlation = np.corrcoef(g_dists, p_dists)[0, 1]
        max_magnitude = torch.norm(final, p=2, dim=-1).max().item()

        active_dims = (torch.var(final, dim=0) > 1e-4).sum().item()

        edges = list(G.edges())
        distortions = [abs(torch.norm(final[u] - final[v], p=2).item() - 1.0) for u, v in edges]
        avg_distortion = np.mean(distortions)

        stability_scores = []
        for node in G.nodes():
            if G.degree(node) == 1 and node != 0:
                parent = list(G.neighbors(node))[0]
                siblings = [n for n in G.neighbors(parent) if n != node]
                if siblings:
                    d_p = torch.norm(final[node] - final[parent], p=2).item()
                    d_s = np.mean([torch.norm(final[node] - final[s], p=2).item() for s in siblings])
                    stability_scores.append(d_s / (d_p + 1e-6))
        avg_stability = np.mean(stability_scores) if stability_scores else 0.0

    return {
        "Correlation": round(correlation, 4),
        "Distortion": round(avg_distortion, 4),
        "Stability": round(avg_stability, 4),
        "Max_Mag": round(max_magnitude, 4),
        "Active_Dims": active_dims,
        "Throughput": int(throughput)
    }

model = EuclideanTransformer(vocab_size, embed_dim, num_heads)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print(f"Training Euclidean Transformer (N={vocab_size})...")
start_train_time = time.time()

for epoch in range(1001):
    model.train()
    optimizer.zero_grad()
    
    all_indices = torch.arange(vocab_size).unsqueeze(0)
    embeddings = model(all_indices).squeeze(0)
    
    u_idx, v_idx = edge_index[:, 0], edge_index[:, 1]
    dist_pos = torch.norm(embeddings[u_idx] - embeddings[v_idx], p=2, dim=-1)
    
    neg_idx = torch.randint(0, vocab_size, (u_idx.size(0),))
    dist_neg = torch.norm(embeddings[u_idx] - embeddings[neg_idx], p=2, dim=-1)
    
    root_penalty = torch.norm(embeddings[0]) 
    
    loss = dist_pos.mean() - 4.0 * torch.log(dist_neg.mean() + 1e-6) + 0.1 * root_penalty
    
    loss.backward()
    optimizer.step()
    
    if epoch % 200 == 0:
        current_elapsed = time.time() - start_train_time
        avg_time = current_elapsed / (epoch + 1)
        print(f"Epoch {epoch:4d} | Loss: {loss.item():.4f} | Total Time: {current_elapsed:.2f}s | Avg: {avg_time:.4f}s/epoch")

total_train_time = time.time() - start_train_time
print(f"\nTraining Complete! Total duration: {total_train_time:.2f} seconds.")

results = evaluate_comprehensive_euclidean(model, G, vocab_size)

print("\n")
print("EUCLIDEAN TRANSFORMER ANALYSIS")
print("~~"*20)
for k, v in results.items():
    print(f"{k:15}: {v}")
print("~~"*20)

Training Euclidean Transformer (N=121)...
Epoch    0 | Loss: -0.5558 | Total Time: 0.00s | Avg: 0.0040s/epoch
Epoch  200 | Loss: -2.7227 | Total Time: 0.60s | Avg: 0.0030s/epoch
Epoch  400 | Loss: -5.4419 | Total Time: 1.24s | Avg: 0.0031s/epoch
Epoch  600 | Loss: -6.5131 | Total Time: 1.91s | Avg: 0.0032s/epoch
Epoch  800 | Loss: -6.9838 | Total Time: 2.55s | Avg: 0.0032s/epoch
Epoch 1000 | Loss: -8.1483 | Total Time: 3.26s | Avg: 0.0033s/epoch

Training Complete! Total duration: 3.26 seconds.

Starting Comprehensive Evaluation (Euclidean Baseline)...


EUCLIDEAN TRANSFORMER ANALYSIS
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Correlation    : 0.2056
Distortion     : 2.0993
Stability      : 1.9892
Max_Mag        : 30.37
Active_Dims    : 16
Throughput     : 121000000000
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~


# Testing with Hierarchic Datasets

In [52]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np
from nltk.corpus import wordnet as wn
import time

class PoincareManifold:
    def __init__(self, c=1.0): self.c = c
    
    def mobius_add(self, x, y):
        x2, y2, xy = torch.sum(x*x, -1, True), torch.sum(y*y, -1, True), torch.sum(x*y, -1, True)
        num = (1 + 2*self.c*xy + self.c*y2)*x + (1 - self.c*x2)*y
        denom = 1 + 2*self.c*xy + self.c**2 * x2*y2
        return num / torch.clamp(denom, min=1e-15)
    
    def exp_map(self, p, v):
        v_norm = torch.norm(v, 2, -1, True).clamp(min=1e-10)
        sqrt_c = np.sqrt(self.c)
        if isinstance(p, (int, float)) and p == 0:
            return torch.tanh(sqrt_c * v_norm) * v / (sqrt_c * v_norm)
        p_norm_sq = torch.sum(p*p, -1, True)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        v_at_origin = torch.tanh(sqrt_c * lambda_p * v_norm / 2) * v / (sqrt_c * v_norm)
        return self.mobius_add(p, v_at_origin)

    def log_map(self, p, x):
        p_tensor = torch.zeros_like(x) if (isinstance(p, (int, float)) and p == 0) else p
        x_trans = self.mobius_add(-p_tensor, x)
        x_norm = torch.norm(x_trans, 2, -1, True).clamp(min=1e-10)
        p_norm_sq = torch.sum(p_tensor*p_tensor, -1, True)
        lambda_p = 2 / (1 - self.c * p_norm_sq).clamp(min=1e-15)
        scalar = (2 / (lambda_p * np.sqrt(self.c) * x_norm)) * torch.atanh((np.sqrt(self.c) * x_norm).clamp(max=1-1e-7))
        return scalar * x_trans

    def dist(self, x, y):
        sq_dist = torch.norm(x - y, 2, -1)**2
        nx_v, ny_v = 1 - self.c * torch.norm(x, 2, -1)**2, 1 - self.c * torch.norm(y, 2, -1)**2
        val = 1 + 2 * self.c * sq_dist / (nx_v.clamp(min=1e-15) * ny_v.clamp(min=1e-15))
        return (1 / np.sqrt(self.c)) * torch.acosh(val.clamp(min=1 + 1e-7))

    def project(self, x):
        norm = torch.norm(x, 2, -1, True)
        return torch.where(norm >= 0.99, x / norm * 0.99, x)

class RSGD(torch.optim.Optimizer):
    def __init__(self, params, lr, manifold): super().__init__(params, dict(lr=lr, manifold=manifold))
    def step(self):
        for group in self.param_groups:
            m, lr = group['manifold'], group['lr']
            for p in group['params']:
                if p.grad is None: continue
                rescale = ((1 - m.c * torch.sum(p.data**2, -1, True))**2 / 4)
                p.data = m.project(m.exp_map(p.data, -lr * p.grad.data * rescale))

class PoincareTransformerBlock(nn.Module):
    def __init__(self, d, h, m):
        super().__init__()
        self.m, self.h, self.d_h = m, h, d // h
        self.q_lin, self.k_lin, self.v_lin = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
        
    def forward(self, x):
        b, s, d = x.shape
        q = self.m.exp_map(0, self.q_lin(self.ln1(x)).view(b, s, self.h, self.d_h)).transpose(1, 2)
        k = self.m.exp_map(0, self.k_lin(self.ln1(x)).view(b, s, self.h, self.d_h)).transpose(1, 2)
        v = self.v_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        
        attn = torch.softmax(-self.m.dist(q.unsqueeze(3), k.unsqueeze(2)) / 1.0, -1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(b, s, d)
        
        x = self.m.exp_map(0, self.m.log_map(0, x) + self.m.exp_map(0, out))
        x = self.m.exp_map(0, self.m.log_map(0, x) + self.ff(self.ln2(x)))
        return x

class DatasetFactory:
    @staticmethod
    def build_from_wordnet(synset_id, max_nodes=500):
        G = nx.Graph()
        try: root = wn.synset(synset_id)
        except: return None
        nodes, visited = [root], {root}
        while nodes and len(G.nodes()) < max_nodes:
            current = nodes.pop(0)
            for child in current.hyponyms():
                if child not in visited:
                    G.add_edge(current.name(), child.name())
                    visited.add(child)
                    nodes.append(child)
        return nx.convert_node_labels_to_integers(G, label_attribute='name')

def evaluate_suite(model, manifold, G, vocab_size, emb_layer):
    with torch.no_grad():
        final = model(manifold.exp_map(0, emb_layer(torch.arange(vocab_size)).unsqueeze(0))).squeeze(0)
        p_dists, g_dists, distortion = [], [], []
        
        for _ in range(min(vocab_size * 2, 1000)):
            u, v = np.random.choice(vocab_size, 2, replace=False)
            try:
                gd = nx.shortest_path_length(G, u, v)
                pd = manifold.dist(final[u], final[v]).item()
                g_dists.append(gd); p_dists.append(pd)
                distortion.append(abs(pd - gd) / max(gd, 1e-6))
            except: continue
            
        stability_vals = []
        for n in G.nodes():
            neighs = list(G.neighbors(n))
            if len(neighs) > 1:
                dists = [manifold.dist(final[n], final[nb]).item() for nb in neighs]
                stability_vals.append(np.mean(dists))
        
        return {
            "Corr": round(np.corrcoef(g_dists, p_dists)[0, 1], 4),
            "Dist": round(np.mean(distortion), 4),
            "Stab": round(np.mean(stability_vals) if stability_vals else 0, 4),
            "Rad":  round(torch.norm(final, p=2, dim=-1).max().item(), 4),
            "Dims": (torch.std(final, dim=0) > 1e-4).sum().item()
        }

domains = [
    ('Balanced Tree', 'SYNTHETIC', 121),
    ('Kinship', 'kinship.n.01', 150),
    ('Living Things', 'living_thing.n.01', 500),
    ('Physics', 'physics.n.01', 100),
    ('Biology', 'biology.n.01', 100),
    ('Mathematics', 'mathematics.n.01', 100)
]

results = {}
for name, synset, limit in domains:
    G = nx.balanced_tree(r=3, h=4) if synset == 'SYNTHETIC' else DatasetFactory.build_from_wordnet(synset, limit)
    if G is None: continue
    
    vocab, dim = len(G.nodes()), 16
    edges = torch.tensor(list(G.edges()), dtype=torch.long)
    manifold = PoincareManifold()
    emb = nn.Embedding(vocab, dim)
    model = PoincareTransformerBlock(dim, 4, manifold)
    opt = RSGD(list(emb.parameters()) + list(model.parameters()), lr=0.5, manifold=manifold)

    t0 = time.time()
    for epoch in range(1201):
        opt.zero_grad()
        x = manifold.exp_map(0, emb(torch.arange(vocab)).unsqueeze(0))
        res = model(x).squeeze(0)
        u, v = edges[:, 0], edges[:, 1]
        loss = manifold.dist(res[u], res[v]).mean() - 1.5 * torch.log(manifold.dist(res[u], res[torch.randint(0, vocab, (u.size(0),))]).mean() + 1e-6)
        loss.backward(); opt.step()
        if epoch == 1000: 
            for g in opt.param_groups: g['lr'] *= 0.1
            
    tps = int(vocab * 1201 / (time.time() - t0))
    metrics = evaluate_suite(model, manifold, G, vocab, emb)
    metrics['TPS'] = tps
    results[name] = metrics

print("\n")
print(f"{'DOMAIN':<18} | {'CORR':<7} | {'DIST':<7} | {'STAB':<7} | {'RAD':<7} | {'DIMS':<5} | {'TPS'}")
print("-" * 80)
for k, v in results.items():
    print(f"{k:<18} | {v['Corr']:<7} | {v['Dist']:<7} | {v['Stab']:<7} | {v['Rad']:<7} | {v['Dims']:<5} | {v['TPS']}")




DOMAIN             | CORR    | DIST    | STAB    | RAD     | DIMS  | TPS
--------------------------------------------------------------------------------
Balanced Tree      | 0.0848  | 0.6414  | 1.6597  | 0.8401  | 16    | 8723
Kinship            | 0.7026  | 1.2026  | 2.0353  | 0.9304  | 16    | 378
Living Things      | 0.1448  | 0.3536  | 1.7283  | 0.8916  | 16    | 9152
Physics            | 0.35    | 0.3807  | 1.6091  | 0.9875  | 16    | 4221
Biology            | 0.2173  | 0.4316  | 1.5557  | 0.8406  | 16    | 5751
Mathematics        | 0.1836  | 0.4666  | 1.5433  | 0.7854  | 16    | 4183


In [54]:
import torch
import torch.nn as nn
import networkx as nx
import numpy as np
from nltk.corpus import wordnet as wn
import time

class EuclideanTransformerBlock(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.h, self.d_h = h, d // h
        self.q_lin, self.k_lin, self.v_lin = nn.Linear(d, d), nn.Linear(d, d), nn.Linear(d, d)
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.ff = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))
        
    def forward(self, x):
        b, s, d = x.shape
        q = self.q_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        k = self.k_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        v = self.v_lin(self.ln1(x)).view(b, s, self.h, self.d_h).transpose(1, 2)
        
        attn = torch.softmax(torch.matmul(q, k.transpose(-2, -1)) / np.sqrt(self.d_h), dim=-1)
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(b, s, d)
        
        x = x + out
        x = x + self.ff(self.ln2(x))
        return x

class DatasetFactory:
    @staticmethod
    def build_from_wordnet(synset_id, max_nodes=500):
        G = nx.Graph()
        try: root = wn.synset(synset_id)
        except: return None
        nodes, visited = [root], {root}
        while nodes and len(G.nodes()) < max_nodes:
            current = nodes.pop(0)
            for child in current.hyponyms():
                if child not in visited:
                    G.add_edge(current.name(), child.name())
                    visited.add(child)
                    nodes.append(child)
        return nx.convert_node_labels_to_integers(G, label_attribute='name')

def evaluate_euclidean(model, G, vocab_size, emb_layer):
    with torch.no_grad():
        final = model(emb_layer(torch.arange(vocab_size)).unsqueeze(0)).squeeze(0)
        p_dists, g_dists, distortion = [], [], []
        
        for _ in range(min(vocab_size * 2, 1000)):
            u, v = np.random.choice(vocab_size, 2, replace=False)
            try:
                gd = nx.shortest_path_length(G, u, v)
                pd = torch.norm(final[u] - final[v], p=2).item()
                g_dists.append(gd); p_dists.append(pd)
                distortion.append(abs(pd - gd) / max(gd, 1e-6))
            except: continue
            
        stability_vals = []
        for n in G.nodes():
            neighs = list(G.neighbors(n))
            if len(neighs) > 1:
                dists = [torch.norm(final[n] - final[nb], p=2).item() for nb in neighs]
                stability_vals.append(np.mean(dists))
        
        return {
            "Corr": round(np.corrcoef(g_dists, p_dists)[0, 1], 4),
            "Dist": round(np.mean(distortion), 4),
            "Stab": round(np.mean(stability_vals) if stability_vals else 0, 4),
            "Rad":  round(torch.norm(final, p=2, dim=-1).max().item(), 4),
            "Dims": (torch.std(final, dim=0) > 1e-4).sum().item()
        }

domains = [
    ('Balanced Tree', 'SYNTHETIC', 121),
    ('Kinship', 'kinship.n.01', 150),
    ('Living Things', 'living_thing.n.01', 500),
    ('Physics', 'physics.n.01', 100),
    ('Biology', 'biology.n.01', 100),
    ('Mathematics', 'mathematics.n.01', 100)
]

euclidean_results = {}
for name, synset, limit in domains:
    G = nx.balanced_tree(r=3, h=4) if synset == 'SYNTHETIC' else DatasetFactory.build_from_wordnet(synset, limit)
    if G is None: continue
    
    vocab, dim = len(G.nodes()), 16
    edges = torch.tensor(list(G.edges()), dtype=torch.long)
    emb = nn.Embedding(vocab, dim)
    model = EuclideanTransformerBlock(dim, 4)
    opt = torch.optim.Adam(list(emb.parameters()) + list(model.parameters()), lr=0.01)

    t0 = time.time()
    for epoch in range(1201):
        opt.zero_grad()
        res = model(emb(torch.arange(vocab)).unsqueeze(0)).squeeze(0)
        u, v = edges[:, 0], edges[:, 1]
        
        pos_dist = torch.norm(res[u] - res[v], p=2, dim=-1)
        neg_dist = torch.norm(res[u] - res[torch.randint(0, vocab, (u.size(0),))], p=2, dim=-1)
        loss = pos_dist.mean() - 1.5 * torch.log(neg_dist.mean() + 1e-6)
        
        loss.backward(); opt.step()
            
    tps = int(vocab * 1201 / (time.time() - t0))
    metrics = evaluate_euclidean(model, G, vocab, emb)
    metrics['TPS'] = tps
    euclidean_results[name] = metrics

print("\n" + "="*30 + " EUCLIDEAN BASELINE RESULTS " + "="*30)
print(f"{'DOMAIN':<18} | {'CORR':<7} | {'DIST':<7} | {'STAB':<7} | {'RAD':<7} | {'DIMS':<5} | {'TPS'}")
print("-" * 85)
for k, v in euclidean_results.items():
    print(f"{k:<18} | {v['Corr']:<7} | {v['Dist']:<7} | {v['Stab']:<7} | {v['Rad']:<7} | {v['Dims']:<5} | {v['TPS']}")
print("="*85)


============================== EUCLIDEAN BASELINE RESULTS ==============================
DOMAIN             | CORR    | DIST    | STAB    | RAD     | DIMS  | TPS
-------------------------------------------------------------------------------------
Balanced Tree      | 0.539   | 10.6743 | 2.6468  | 93.4206 | 16    | 41252
Kinship            | 0.9646  | 0.7965  | 1.9021  | 16.3016 | 16    | 1661
Living Things      | 0.7361  | 111.3815 | 1.3471  | 693.9869 | 16    | 94580
Physics            | 0.2885  | 8.1442  | 0.9939  | 65.5235 | 16    | 18477
Biology            | 0.1687  | 7.7813  | 0.7624  | 104.8111 | 16    | 25392
Mathematics        | 0.2084  | 6.4686  | 1.183   | 60.0803 | 16    | 18606
